# 🏋️ Exercise Classification — Model Training

This notebook trains a **RandomForestClassifier** on the extracted keypoint features
to classify exercises and detect form correctness.

**Classes:** `squat_correct`, `squat_incorrect`, `pushup_correct`, `pushup_incorrect`,
`lunge_correct`, `lunge_incorrect`, `plank_correct`, `plank_incorrect`

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

print('All imports successful ✅')

## 2. Load Dataset

If you haven't generated the dataset yet, run one of:
```bash
python dataset/generate_synthetic.py   # synthetic data (no videos needed)
python dataset/prepare_dataset.py      # from real videos
```

In [ ]:
# Path to the keypoints CSV
CSV_PATH = os.path.join('..', 'dataset', 'processed_keypoints', 'keypoints.csv')

# For Colab: uncomment the line below if the notebook is in /content/
# CSV_PATH = '/content/AI-Exercise-Detection/dataset/processed_keypoints/keypoints.csv'

df = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Classes: {df["label"].nunique()}')
df['label'].value_counts()

## 3. Prepare Features and Labels

In [ ]:
X = df.drop(columns=['label']).values
y = df['label'].values

# Stratified split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')

## 4. Train RandomForestClassifier

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)

clf.fit(X_train, y_train)
print('Training complete ✅')

## 5. Evaluate the Model

In [ ]:
y_pred = clf.predict(X_test)

# Overall metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {f1:.4f}')
print()
print('Per-class report:')
print(classification_report(y_test, y_pred))

## 6. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title('Exercise Classification — Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = clf.feature_importances_
feature_names = df.drop(columns=['label']).columns.tolist()

# Top 15 features
indices = np.argsort(importances)[-15:]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(indices)), importances[indices], color='teal')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices])
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

## 8. Save Trained Model

In [ ]:
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, 'exercise_model.pkl')
joblib.dump(clf, MODEL_PATH)
print(f'Model saved to {MODEL_PATH} ✅')

## 9. Quick Sanity Check — Load & Predict

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

# Predict on a random test sample
idx = np.random.randint(len(X_test))
sample = X_test[idx].reshape(1, -1)
pred = loaded_model.predict(sample)[0]
actual = y_test[idx]
print(f'Sample #{idx}  |  Predicted: {pred}  |  Actual: {actual}')

---
**Done!** The trained model is saved at `models/exercise_model.pkl`.

Next steps:
- Run `python demo/run_webcam_demo.py` for real-time inference.
- See `src/form_analyzer.py` for rule-based form checks.